In [74]:
read_data <- function(path) {
  d <- read.csv(path, stringsAsFactors = FALSE)
  d <- d[, c("Azimuth", "Lat", "Lon", "GpsLat", "GpsLon", "Session")]
  d <- d[!is.na(d$Azimuth),]
  d$point <- seq_len(nrow(d))
  d
}

In [75]:
d <- read_data("resection_log_golden3.csv")

In [76]:
View(d)

,Azimuth,Lat,Lon,GpsLat,GpsLon,Session,point
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
1,295.3332,-16.38715,-71.52203,-16.39161,-71.51223,S04,1
2,97.0792,-16.39440,-71.48876,-16.39161,-71.51223,S04,2
3,43.4447,-16.16101,-71.28495,-16.39161,-71.51223,S04,3
4,169.1166,-16.45733,-71.49905,-16.39161,-71.51223,S04,4


In [78]:
resection <- function(az, x, y, add = rep(TRUE, length(az))) {
  az <- az[add]
  x <- x[add]
  y <- y[add]
  a <- az * rad
  nx <- cos(a)
  ny <- -sin(a)
  b <- nx * x + ny * y
  A <- matrix(c(sum(nx^2), sum(nx * ny), sum(nx * ny), sum(ny^2)), 2, 2)
  p <- solve(A, c(sum(nx * b), sum(ny * b)))
  rex <- nx * (p[1] - x) + ny * (p[2] - y)
  list(east = p[1], north = p[2], error = sqrt(sum(p^2)))
}


In [79]:
RADIUS_EARTH <- 6371008.0
rad <- pi / 180

In [80]:
to_degrees <- function(x, y, origin) {
  c(
    lat = origin[["lat"]] + y / (rad * RADIUS_EARTH),
    lon = origin[["lon"]] +
      x / (rad * RADIUS_EARTH * cos(origin[["lat"]] * rad))
  )
}

to_meters <- function(lat, lon, origin) {
  x <- (lon - origin[["lon"]]) *
    rad *
    RADIUS_EARTH *
    cos(origin[["lat"]] * rad)
  y <- (lat - origin[["lat"]]) * rad * RADIUS_EARTH
  cbind(x = x, y = y)
}


In [81]:
run <- function(table, origin, k = 4, daz_val = 1, dpos_val = 0.0001) {
  t <- table[seq_len(k),]
  n <- nrow(t)

  row_resecton <- function(
    name,
    daz = rep(0, n),
    dpos = rep(0, n),
    add = rep(TRUE, n)
  ) {
    lat <- t$lat + dpos
    lon <- t$lon + dpos
    az <- t$az + daz
    xy <- to_meters(lat, lon, origin)
    s <- resection(az, xy[, "x"], xy[, "y"], add)
    sol <- to_degrees(s$east, s$north, origin)
    P <- ifelse(add, round(az, 2), NA)
    Q <- ifelse(add, sprintf("(%.6f, %.6f)", lat, lon), NA)
    out <- as.data.frame(
      c(
        list(scenario = name),
        setNames(as.list(P), paste0("P", 1:n)),
        setNames(as.list(Q), paste0("Q", 1:n))
      ),
      stringAsFactors = FALSE
    )

    out$Result1 <- sprintf("(%.7f, %.7f)", sol[["lat"]], sol[["lon"]])
    out$Result2 <- sprintf("(%.7f, %.7f)", origin[["lat"]], origin[["lon"]])
    out$error_origin_m <- s$error
    out
  }

  r <- row_resecton("BASE")
  for (i in 1:n) {
    d <- rep(0, n)
    d[i] <- daz_val
    r <- rbind(r, row_resecton(paste0("+1 Azimuth angle in P", i), daz = d))
  }
  r <- rbind(r, row_resecton("+1 Azimuth angle in all", daz = rep(daz_val, n)))
  for (i in 1:n) {
    d <- rep(0, n)
    d[i] <- dpos_val
    r <- rbind(r, row_resecton(paste0("+0.0001 lat/lon in P", i), dpos = d))
  }
  r <- rbind(r, row_resecton("+0.0001 lat/lon in all", dpos = rep(dpos_val, n)))
  if (n > 2) {
    for (i in 1:n) {
      r <- rbind(
        r,
        row_resecton(paste0("Without P", i), add = seq_len(n) != i)
      )
    }
  }
  r
}

In [82]:
# origin <- c(lat = 48.7022484, lon = 2.1530186)
origin <- c(lat = d$GpsLat[1], lon = d$GpsLon[1])
table <- data.frame(point = d$point, az = d$Azimuth, lat = d$Lat, lon = d$Lon)

res <- run(table, origin, k = 4)

In [83]:
View(res)

scenario,P1,P2,P3,P4,Q1,Q2,Q3,Q4,Result1,Result2,error_origin_m
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
BASE,295.33,97.08,43.44,169.12,"(-16.387154, -71.522031)","(-16.394402, -71.488760)","(-16.161010, -71.284955)","(-16.457326, -71.499051)","(-16.3915153, -71.5123554)","(-16.3916072, -71.5122261)",17.1642158
+1 Azimuth angle in P1,296.33,97.08,43.44,169.12,"(-16.387154, -71.522031)","(-16.394402, -71.488760)","(-16.161010, -71.284955)","(-16.457326, -71.499051)","(-16.3915815, -71.5123959)","(-16.3916072, -71.5122261)",18.3357232
+1 Azimuth angle in P2,295.33,98.08,43.44,169.12,"(-16.387154, -71.522031)","(-16.394402, -71.488760)","(-16.161010, -71.284955)","(-16.457326, -71.499051)","(-16.3913452, -71.5123439)","(-16.3916072, -71.5122261)",31.7269501
+1 Azimuth angle in P3,295.33,97.08,44.44,169.12,"(-16.387154, -71.522031)","(-16.394402, -71.488760)","(-16.161010, -71.284955)","(-16.457326, -71.499051)","(-16.3896325, -71.5150476)","(-16.3916072, -71.5122261)",372.5609249
+1 Azimuth angle in P4,295.33,97.08,43.44,170.12,"(-16.387154, -71.522031)","(-16.394402, -71.488760)","(-16.161010, -71.284955)","(-16.457326, -71.499051)","(-16.3914814, -71.5116505)","(-16.3916072, -71.5122261)",62.9762937
+1 Azimuth angle in all,296.33,98.08,44.44,170.12,"(-16.387154, -71.522031)","(-16.394402, -71.488760)","(-16.161010, -71.284955)","(-16.457326, -71.499051)","(-16.3894716, -71.5143495)","(-16.3916072, -71.5122261)",328.1731973
+0.0001 lat/lon in P1,295.33,97.08,43.44,169.12,"(-16.387054, -71.521931)","(-16.394402, -71.488760)","(-16.161010, -71.284955)","(-16.457326, -71.499051)","(-16.3914662, -71.5123266)","(-16.3916072, -71.5122261)",18.9922820
+0.0001 lat/lon in P2,295.33,97.08,43.44,169.12,"(-16.387154, -71.522031)","(-16.394302, -71.488660)","(-16.161010, -71.284955)","(-16.457326, -71.499051)","(-16.3914678, -71.5123527)","(-16.3916072, -71.5122261)",20.5541605
+0.0001 lat/lon in P3,295.33,97.08,43.44,169.12,"(-16.387154, -71.522031)","(-16.394402, -71.488760)","(-16.160910, -71.284855)","(-16.457326, -71.499051)","(-16.3915156, -71.5123549)","(-16.3916072, -71.5122261)",17.1072461


In [98]:
library(ggplot2)
library(ggrepel)

AZUL <- "#2a78d6"
NARANJA <- "#eb6834"
VERDE <- "#1baf7a"
GRIS <- "#8a8a8a"
dir.create("charts", showWarnings = FALSE)


# ---------- datos comunes ----------
xy <- to_meters(table$lat, table$lon, origin)
pines <- data.frame(
  punto = paste0("P", seq_len(nrow(table))),
  x = xy[, "x"], y = xy[, "y"], az = table$az
)

pines$punto      # debe mostrar "P1" "P2" "P3"

parse_ll <- function(s) {
  n <- as.numeric(strsplit(gsub("[()]", "", s), ", ")[[1]])
  c(lat = n[1], lon = n[2])
}

sol_m <- t(sapply(res$Result1, function(s) {
  p <- parse_ll(s)
  to_meters(p[["lat"]], p[["lon"]], origin)
}))
dimnames(sol_m) <- list(NULL, c("x", "y"))
base <- sol_m[1,]


# ---------- 1. MAPA EN METROS ----------
L <- 2 * max(sqrt(pines$x^2 + pines$y^2))
LBL_REAL <- "True direction (origin → point)"
LBL_AZ <- "Measured azimuth line"

rayos <- transform(pines, dx = sin(az * pi / 180), dy = cos(az * pi / 180))
rayos$x0 <- rayos$x - L * rayos$dx
rayos$y0 <- rayos$y - L * rayos$dy
rayos$x1 <- rayos$x + L * rayos$dx
rayos$y1 <- rayos$y + L * rayos$dy

radios <- 300 + 250 * seq_len(nrow(pines))
arcos <- do.call(rbind, lapply(seq_len(nrow(pines)), function(i) {
  a <- seq(0, pines$az[i], length.out = 80) * pi / 180
  data.frame(punto = pines$punto[i], x = radios[i] * sin(a), y = radios[i] * cos(a))
}))

# Etiqueta al final de cada arco
etq <- data.frame(
  x = radios * sin(pines$az * pi / 180),
  y = radios * cos(pines$az * pi / 180),
  txt = sprintf("%s: %.1f°", pines$punto, pines$az),
  az = pines$az
)

todos_x <- c(pines$x, 0, base[["x"]])
todos_y <- c(pines$y, 0, base[["y"]])
mx <- 0.12 * diff(range(todos_x))
my <- 0.12 * diff(range(todos_y))

g1 <- ggplot() +
  geom_segment(data = rayos, aes(x = x0, y = y0, xend = x1, yend = y1, linetype = LBL_AZ, colour = LBL_AZ),
               linewidth = 0.7) +
  geom_segment(
    data = pines,
    aes(
      x = 0,
      y = 0,
      xend = x,
      yend = y,
      linetype = LBL_REAL,
      colour = LBL_REAL
    ),
    linewidth = 0.5
  ) +
  geom_segment(
    aes(x = 0, y = 0, xend = base[["x"]], yend = base[["y"]]),
    colour = NARANJA,
    linewidth = 0.3,
    arrow = arrow(length = unit(0.25, "cm"))
  ) +
  # línea de referencia Norte
  annotate("segment", x = 0, y = 0, xend = 0, yend = max(radios) + 150,
           colour = "black", linewidth = 0.4) +
  annotate("text", x = 0, y = max(radios) + 230, label = "N",
           fontface = "bold", size = 3.6) +

  # arcos del ángulo azimut
  geom_path(data = arcos, aes(x, y, group = punto),
            colour = NARANJA, linewidth = 0.8,
            arrow = arrow(length = unit(0.12, "cm"), type = "closed")) +
  geom_text(data = etq,
            aes(x, y, label = txt,
                hjust = 1.1,
                vjust = 0.5),
            colour = NARANJA, size = 3.3, fontface = "bold") +
  geom_point(data = pines, aes(x, y), size = 3) +
  geom_text_repel(data = pines, aes(x, y, label = punto), fontface = "bold") +
  geom_point(aes(x = 0, y = 0), shape = 8, size = 3.5) +
  geom_point(aes(x = base[["x"]], y = base[["y"]]), colour = AZUL, size = 3.5) +
  annotate("label", x = 1500, y = 2700,
           label = sprintf("Solution (BASE)\nerror %.0f m", res$error_origin_m[1]),
           colour = AZUL, fontface = "bold", size = 3.5, fill = "white", border.colour = NA) +
  annotate("segment", x = 1350, y = 2500,
           xend = base[["x"]] - 25, yend = base[["y"]] + 25,
           colour = AZUL, linewidth = 0.4) +
  annotate("label", x = 1300, y = -1700, label = "Origin",
           size = 3.5, fill = "white", border.colour = NA) +
  annotate("segment", x = 1200, y = -1600, xend = -25, yend = 25,
           colour = "black", linewidth = 0.3) +

  # UNA sola leyenda: colour y linetype con los mismos nombres
  scale_colour_manual(values = setNames(c(GRIS, VERDE), c(LBL_REAL, LBL_AZ)), name = NULL) +
  scale_linetype_manual(values = setNames(c("solid", "dashed"), c(LBL_REAL, LBL_AZ)), name = NULL) +
  coord_fixed(xlim = c(-2000, 6500), ylim = c(-5000, 3000)) +
  labs(
    title = "Map in meters: azimuth angles (from north, clockwise)",
    x = "East (m)",
    y = "North (m)"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom", plot.title = element_text(face = "bold"))

[1] "P1" "P2" "P3" "P4"

In [99]:
ggsave("charts/1_mapa_brazil1.png", g1, width = 9, height = 7, dpi = 150)


In [100]:

pines$rumbo_real <- (atan2(pines$x, pines$y) * 180 / pi) %% 360
pines$error_ang <- ((pines$az - pines$rumbo_real + 180) %% 360) - 180   # entre -180 y 180

g2 <- ggplot(pines, aes(punto, error_ang)) +
  geom_hline(yintercept = 0, colour = "black") +
  geom_col(fill = AZUL, width = 0.55) +
  geom_text(aes(label = sprintf("%+.1f°", error_ang), vjust = ifelse(error_ang >= 0, -0.4, 1.4)), fontface = "bold") +
  scale_y_continuous(expand = expansion(mult = 0.15)) +
  labs(title = "Error angular por punto (azimut medido − rumbo real)",
       subtitle = "negativo = apuntaste a la izquierda del pin; positivo = a la derecha", x = NULL, y = "error (grados)") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"), panel.grid.major.x = element_blank())
ggsave("charts/2_error_angular.png", g2, width = 7, height = 5, dpi = 150)


In [101]:
# ---------- 3. SENSIBILIDAD ----------
sens <- data.frame(scenario = res$scenario, x = sol_m[, "x"], y = sol_m[, "y"], error = res$error_origin_m)
sens$delta_m <- sqrt((sens$x - base[["x"]])^2 + (sens$y - base[["y"]])^2)
sens$grupo <- ifelse(grepl("Azimuth", sens$scenario), "Azimut change",
                     ifelse(grepl("lat/lon", sens$scenario), "Point change", "Without a Point"))
sens <- sens[sens$scenario != "BASE",]
sens$scenario <- factor(sens$scenario, levels = rev(sens$scenario))

g3 <- ggplot(sens, aes(delta_m, scenario, colour = grupo)) +
  geom_segment(aes(x = 0.5, xend = delta_m, yend = scenario), linewidth = 0.8) +
  geom_point(size = 2.4) +
  geom_text(aes(label = sprintf("%.2f m", delta_m)), hjust = -0.35, size = 3.5, show.legend = FALSE) +
  scale_x_log10(limits = c(0.5, max(sens$delta_m) * 4),
                breaks = c(0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000),
                labels = c("0.1", "0.5", "1", "5", "10", "50", "100", "500", "1000")) +
  scale_colour_manual(values = c("Azimut change" = AZUL, "Point change" = NARANJA, "Without a Point" = VERDE), name = NULL) +
  labs(title = "Sensitivity: how far the solution shifts from the BASE/origin",
       x = "Displacement of the solution (m, logarithmic scale)", y = NULL) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom", plot.title = element_text(face = "bold"), panel.grid.minor = element_blank())
ggsave("charts/2_sensibilidad.png", g3, width = 9, height = 6, dpi = 150)
print(pines[, c("punto", "az", "rumbo_real", "error_ang")]); print(sens[, c("scenario", "delta_m", "error")])


Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_segment()`)."
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_point()`)."
Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_text()`)."


  punto       az rumbo_real    error_ang
1    P1 295.3332  295.33483 -0.001634708
2    P2  97.0792   97.07586  0.003343453
3    P3  43.4447   43.39594  0.048761984
4    P4 169.1166  169.11296  0.003641024
                  scenario      delta_m       error
2   +1 Azimuth angle in P1   8.53530954  18.3357232
3   +1 Azimuth angle in P2  18.95402278  31.7269501
4   +1 Azimuth angle in P3 355.40124162 372.5609249
5   +1 Azimuth angle in P4  75.29003785  62.9762937
6  +1 Azimuth angle in all 311.27611898 328.1731973
7    +0.0001 lat/lon in P1   6.26473002  18.9922820
8    +0.0001 lat/lon in P2   5.28961305  20.5541605
9    +0.0001 lat/lon in P3   0.06291034  17.1072461
10   +0.0001 lat/lon in P4   7.27688495  12.4785937
11  +0.0001 lat/lon in all  15.40909532  21.5659898
12              Without P1   2.47635764  17.6445117
13              Without P2   6.20296523  21.2179398
14              Without P3  17.53246094   0.4409513
15              Without P4  16.36936856  31.5373931
